# Arquitectura OLAP en Spotify (Dataset Real 2024) 🎵
## Contexto:
Eres el Arquitecto de Datos Principal en una disquera. El equipo de ingeniería extrajo un dump transaccional masivo con las canciones más reproducidas de Spotify en 2024. El archivo plano es ancho, redundante y poco óptimo para análisis.

Tu misión es aplicar una tubería ETL para convertir este archivo crudo en un Esquema de Estrella en memoria, y posteriormente responder a 10 preguntas críticas del equipo de analítica utilizando el Cubo OLAP.

In [2]:
import pandas as pd
import numpy as np

# EXTRACT: Leemos directamente el archivo crudo desde el repositorio
url_dataset = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"

# Para evitar errores de codificación con caracteres especiales en nombres de artistas
spotify_raw = pd.read_csv(url_dataset, encoding='latin-1')

print("Muestra de datos crudos (OLTP):")
display(spotify_raw.head(3))
print("Columnas disponibles:", spotify_raw.columns.tolist())

Muestra de datos crudos (OLTP):


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0


Columnas disponibles: ['Track', 'Album Name', 'Artist', 'Release Date', 'ISRC', 'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'Shazam Counts', 'TIDAL Popularity', 'Explicit Track']


### Fase 2: Transform & Load (Construyendo el Almacén)

Observa las columnas del dataset. Tienes texto (Track, Artist, Date) mezclado con métricas puras (Spotify Streams, YouTube Views). 

**Reto Arquitectónico:**
1. Crea una **Dimensión Tiempo (`dim_tiempo`)**: Extrae las columnas relacionadas al año, mes y día de lanzamiento. Elimina duplicados y crea un `id_tiempo`.
2. Crea una **Dimensión Pista (`dim_pista`)**: Extrae el nombre del track, el artista y si es explícita. Crea un `id_pista`.
3. Crea la **Tabla de Hechos (`fact_streams`)**: Cruza los IDs y conserva **únicamente** métricas numéricas (ej. Spotify Streams, YouTube Views, Track Score) junto con las llaves foráneas. NO debe haber texto aquí.

In [3]:
# --- ESCRIBE TU PIPELINE ETL AQUÍ ---

# 1. Crear dim_tiempo
dim_tiempo = spotify_raw[['Release Date']].drop_duplicates().reset_index(drop = True)
dim_tiempo['id_tiempo'] = dim_tiempo['id_tiempo'] = range(1, len(dim_tiempo) + 1)

dim_tiempo['Release Date'] = pd.to_datetime(
    dim_tiempo['Release Date'],
    errors='coerce'
)

dim_tiempo['dia'] = dim_tiempo['Release Date'].dt.day
dim_tiempo['mes'] = dim_tiempo['Release Date'].dt.month
dim_tiempo['año'] = dim_tiempo['Release Date'].dt.year

print("Dimensión Tiempo (Muestra):")
display(dim_tiempo.head(3))

Dimensión Tiempo (Muestra):


,Release Date,id_tiempo,dia,mes,año
0,2024-04-26,1,26,4,2024
1,2024-05-04,2,4,5,2024
2,2024-03-19,3,19,3,2024


In [4]:
# 2. Crear dim_pista
# .copy() evita el SettingWithCopyWarning y drop_duplicates evita que el merge multiplique filas
dim_pista = spotify_raw[['Track', 'Artist', 'Explicit Track']].drop_duplicates().reset_index(drop=True).copy()
dim_pista['id_pista'] = range(1, len(dim_pista) + 1)

print("Dimensión Pista (Muestra):")
display(dim_pista.head(3))

Dimensión Pista (Muestra):


,Track,Artist,Explicit Track,id_pista
0,MILLION DOLLAR BABY,Tommy Richman,0,1
1,Not Like Us,Kendrick Lamar,1,2
2,i like the way you kiss me,Artemas,0,3


In [5]:
# 3. Crear fact_streams
spotify_raw['Release Date'] = pd.to_datetime(spotify_raw['Release Date'], errors='coerce')

# Cruzamos con las dimensiones usando la llave completa de cada una
df_temp = spotify_raw.merge(dim_pista, on=['Track', 'Artist', 'Explicit Track'], how='left')
df_temp = df_temp.merge(dim_tiempo[['Release Date', 'id_tiempo']], on='Release Date', how='left')

metricas = ['Spotify Streams', 'YouTube Views', 'Track Score', 'Pandora Streams', 'TikTok Views', 'Shazam Counts']

fact_streams = df_temp[['id_pista', 'id_tiempo'] + metricas].copy()

# Las métricas vienen como texto con comas ("390,470,936"), las convertimos a número
for col in metricas:
    fact_streams[col] = pd.to_numeric(
        fact_streams[col].astype(str).str.replace(',', ''),
        errors='coerce'
    )

print("Tabla de Hechos (Métrica Pura):")
display(fact_streams.head(3))
print(fact_streams.dtypes)

Tabla de Hechos (Métrica Pura):


,id_pista,id_tiempo,Spotify Streams,YouTube Views,Track Score,Pandora Streams,TikTok Views,Shazam Counts
0,1,1,390470936.0,84274754.0,725.4,18004655.0,5.332282e+09,2669262.0
1,2,2,323703884.0,116347040.0,545.9,7780028.0,2.083390e+08,1118279.0
2,3,3,601309283.0,122599116.0,538.4,5022621.0,3.369121e+09,5285340.0


id_pista             int64
id_tiempo            int64
Spotify Streams    float64
YouTube Views      float64
Track Score        float64
Pandora Streams    float64
TikTok Views       float64
Shazam Counts      float64
dtype: object


In [6]:
# 4. (LOAD) Reconstruir el Cubo en memoria para las consultas
cubo_olap = (
    fact_streams
    .merge(dim_pista, on='id_pista', how='left')
    .merge(dim_tiempo, on='id_tiempo', how='left')
)

print("Cubo OLAP:", cubo_olap.shape)
display(cubo_olap.head(3))

Cubo OLAP: (4600, 15)


,id_pista,id_tiempo,Spotify Streams,YouTube Views,Track Score,Pandora Streams,TikTok Views,Shazam Counts,Track,Artist,Explicit Track,Release Date,dia,mes,año
0,1,1,390470936.0,84274754.0,725.4,18004655.0,5.332282e+09,2669262.0,MILLION DOLLAR BABY,Tommy Richman,0,2024-04-26,26,4,2024
1,2,2,323703884.0,116347040.0,545.9,7780028.0,2.083390e+08,1118279.0,Not Like Us,Kendrick Lamar,1,2024-05-04,4,5,2024
2,3,3,601309283.0,122599116.0,538.4,5022621.0,3.369121e+09,5285340.0,i like the way you kiss me,Artemas,0,2024-03-19,19,3,2024


### Explica porqué solo hay 2 tablas dimensiones o si podrías incluir alguna otra. 

Actuialmente el proposito de las dimensiopnes es que nos repsondan preguntas descriptivas como ¿Quien? ¿Que?, ¿Cuando?, en donde en este caso tenemos marcados dos dimensiones que es la de tiempo (Release Date) y la de que producto fue el que cancion es. Como todo lo demas son datos numericos no correspondian a una tabla. 

Aunque si se pódria agregar otra dimension que repsonde al ¿Quien? se puede referir al artista ciomno tal para no tener que repetir el nombre en cada cancion que sea del mismo.


### Fase 3: Consultas Analíticas (10 Retos)

El equipo de analítica te ha enviado 10 peticiones. Para cada una, debes:
1. Escribir el código en Pandas usando la matriz `cubo_olap` que construiste.
2. Identificar y escribir en un comentario qué tipo de operación OLAP estás aplicando: **Roll-up, Drill-down, Slice o Dice**.

In [7]:
# RETO 1: El CEO quiere ver el total acumulado de 'Spotify Streams' agrupado por 'Artist'.
# Operación OLAP: Roll-up (se agregan las canciones al nivel de artista)
respuesta_1 = cubo_olap.groupby('Artist')['Spotify Streams'].sum().sort_values(ascending=False)
print("RETO 1"); display(respuesta_1.head(10))

# RETO 2: El equipo de marketing quiere ver todas las métricas únicamente de las canciones lanzadas en el año 2024.
# Operación OLAP: Slice (se fija una sola dimensión: año = 2024)
respuesta_2 = cubo_olap[cubo_olap['año'] == 2024]
print("RETO 2"); display(respuesta_2.head())

# RETO 3: Finanzas requiere el total de 'Spotify Streams' desglosado primero por Año de Lanzamiento (Year) y luego detallado por Mes de Lanzamiento (Month).
# Operación OLAP: Drill-down (de año baja al nivel más detallado de mes)
respuesta_3 = cubo_olap.groupby(['año', 'mes'])['Spotify Streams'].sum()
print("RETO 3"); display(respuesta_3.tail(12))

# RETO 4: Auditoría te pide extraer el rendimiento de las canciones que son explícitas Y que además fueron lanzadas en el año 2024.
# Operación OLAP: Dice (filtro sobre dos dimensiones: pista y tiempo)
respuesta_4 = cubo_olap[(cubo_olap['Explicit Track'] == 1) & (cubo_olap['año'] == 2024)]
print("RETO 4"); display(respuesta_4.head())

# RETO 5: Queremos disminuir el nivel de detalle y ver solamente el promedio del 'Track Score' agrupado por Año de Lanzamiento.
# Operación OLAP: Roll-up (se sube de nivel en la jerarquía de tiempo, a año)
respuesta_5 = cubo_olap.groupby('año')['Track Score'].mean()
print("RETO 5"); display(respuesta_5)

RETO 1


Artist
Bad Bunny         3.705483e+10
The Weeknd        3.694854e+10
Drake             3.496216e+10
Taylor Swift      3.447077e+10
Post Malone       2.613747e+10
Ed Sheeran        2.401490e+10
Ariana Grande     2.346499e+10
MUSIC LAB JPN     2.286669e+10
Olivia Rodrigo    1.972922e+10
Eminem            1.887888e+10
Name: Spotify Streams, dtype: float64

RETO 2


,id_pista,id_tiempo,Spotify Streams,YouTube Views,Track Score,Pandora Streams,TikTok Views,Shazam Counts,Track,Artist,Explicit Track,Release Date,dia,mes,año
0,1,1,390470936.0,84274754.0,725.4,18004655.0,5.332282e+09,2669262.0,MILLION DOLLAR BABY,Tommy Richman,0,2024-04-26,26,4,2024
1,2,2,323703884.0,116347040.0,545.9,7780028.0,2.083390e+08,1118279.0,Not Like Us,Kendrick Lamar,1,2024-05-04,4,5,2024
2,3,3,601309283.0,122599116.0,538.4,5022621.0,3.369121e+09,5285340.0,i like the way you kiss me,Artemas,0,2024-03-19,19,3,2024
4,5,5,107034922.0,77373957.0,423.3,4493884.0,NaN,457017.0,Houdini,Eminem,1,2024-05-31,31,5,2024
6,7,7,900158751.0,308723145.0,407.2,65447476.0,5.349153e+08,9990302.0,Beautiful Things,Benson Boone,0,2024-01-18,18,1,2024


RETO 3


año   mes
2023  7      2.376234e+10
      8      2.041237e+10
      9      1.649861e+10
      10     2.403831e+10
      11     1.723808e+10
      12     8.333585e+09
2024  1      1.224484e+10
      2      1.023077e+10
      3      1.263673e+10
      4      2.152035e+10
      5      1.005348e+10
      6      2.711085e+09
Name: Spotify Streams, dtype: float64

RETO 4


,id_pista,id_tiempo,Spotify Streams,YouTube Views,Track Score,Pandora Streams,TikTok Views,Shazam Counts,Track,Artist,Explicit Track,Release Date,dia,mes,año
1,2,2,3.237039e+08,116347040.0,545.9,7780028.0,2.083390e+08,1118279.0,Not Like Us,Kendrick Lamar,1,2024-05-04,4,5,2024
4,5,5,1.070349e+08,77373957.0,423.3,4493884.0,NaN,457017.0,Houdini,Eminem,1,2024-05-31,31,5,2024
7,8,8,6.750792e+08,228382568.0,375.8,3372428.0,3.804584e+09,6063523.0,Gata Only,FloyyMenor,1,2024-02-02,2,2,2024
8,9,9,1.653018e+09,NaN,355.7,NaN,NaN,NaN,Danza Kuduro - Cover,MUSIC LAB JPN,1,2024-06-09,9,6,2024
9,10,10,9.067657e+07,32735244.0,330.6,1005626.0,9.746562e+08,666302.0,BAND4BAND (feat. Lil Baby),Central Cee,1,2024-05-23,23,5,2024


RETO 5


año
1987    21.500000
1991    19.600000
1994    31.500000
1998    20.800000
1999    22.200000
2000    30.400000
2001    20.100000
2002    26.060000
2003    23.766667
2004    24.240000
2005    27.140000
2006    22.300000
2007    23.860000
2008    23.854545
2009    26.000000
2010    31.425926
2011    31.800000
2012    32.197872
2013    31.674545
2014    31.678571
2015    37.988542
2016    37.828455
2017    37.949765
2018    36.977510
2019    39.380405
2020    38.042222
2021    38.144118
2022    41.693228
2023    43.264594
2024    54.120058
Name: Track Score, dtype: float64

In [8]:
# RETO 6: Muestra el comportamiento analítico aislando únicamente los datos del artista "Bad Bunny".
# Operación OLAP: Slice (se fija una sola dimensión: Artist = Bad Bunny)
respuesta_6 = cubo_olap[cubo_olap['Artist'] == 'Bad Bunny']
print("RETO 6"); display(respuesta_6.head())

# RETO 7: Aumenta el nivel de detalle: Muestra el total de 'YouTube Views' agrupado por Artista, y dentro de cada Artista, desglosado por Nombre de la Canción (Track).
# Operación OLAP: Drill-down (de artista baja al nivel de canción)
respuesta_7 = cubo_olap.groupby(['Artist', 'Track'])['YouTube Views'].sum()
print("RETO 7"); display(respuesta_7.head(10))

# RETO 8: Extrae un sub-cubo que contenga únicamente canciones lanzadas en el Mes 12 (Diciembre) del Año 2023.
# Operación OLAP: Dice (sub-cubo filtrando por dos niveles/valores de la dimensión tiempo)
respuesta_8 = cubo_olap[(cubo_olap['mes'] == 12) & (cubo_olap['año'] == 2023)]
print("RETO 8"); display(respuesta_8.head())

# RETO 9: Resume la información calculando el total histórico de 'Spotify Streams' de todos los tiempos, agrupado por Mes (para saber en qué mes históricamente se lanza la música más exitosa).
# Operación OLAP: Roll-up (se agregan todos los años y se resume por mes)
respuesta_9 = cubo_olap.groupby('mes')['Spotify Streams'].sum().sort_values(ascending=False)
print("RETO 9"); display(respuesta_9)

# RETO 10: Extrae un bloque de información filtrando las canciones de "Taylor Swift" que NO sean explícitas y hayan salido en 2024.
# Operación OLAP: Dice (filtro en varias dimensiones: artista, explícita y año)
respuesta_10 = cubo_olap[
    (cubo_olap['Artist'] == 'Taylor Swift') &
    (cubo_olap['Explicit Track'] == 0) &
    (cubo_olap['año'] == 2024)
]
print("RETO 10"); display(respuesta_10)

RETO 6


,id_pista,id_tiempo,Spotify Streams,YouTube Views,Track Score,Pandora Streams,TikTok Views,Shazam Counts,Track,Artist,Explicit Track,Release Date,dia,mes,año
63,61,57,1.811991e+09,1.516863e+09,184.8,111324744.0,6.470937e+09,3393920.0,Me Porto Bonito,Bad Bunny,1,2022-05-06,6,5,2022
88,86,57,1.514240e+09,1.366181e+09,163.1,69095169.0,5.249864e+09,5133847.0,Titï¿½ï¿½ Me Pregu,Bad Bunny,1,2022-05-06,6,5,2022
102,100,91,4.781509e+08,1.133797e+08,150.7,17823388.0,2.111289e+09,2255465.0,MONACO,Bad Bunny,1,2023-10-13,13,10,2023
174,171,91,6.891815e+08,1.833815e+08,114.2,14039085.0,4.874404e+08,930098.0,PERRO NEGRO,Bad Bunny,1,2023-10-13,13,10,2023
187,184,57,1.421763e+09,9.458571e+08,110.7,57148186.0,2.251428e+08,3087489.0,Ojitos Lindos,Bad Bunny,0,2022-05-06,6,5,2022


RETO 7


Artist       Track                                                              
"XY"         Come and Say - Original Mix                                            415159148.0
$OHO BANI    ZEIT, DASS SICH WAS DREHT                                                6744201.0
$uicideboy$  ...And to Those I Love, Thanks for Sticking Around                     269460083.0
             1000 Blunts                                                             87680985.0
             Are You Going to See the Rose in the Vase, or the Dust on the Table      2640169.0
             Avalon                                                                  49001959.0
             For the Last Time                                                      149247747.0
             Matte Black                                                             82186979.0
             Not Even Ghosts Are This Empty                                          50747796.0
             The Thin Grey Line                        

RETO 8


,id_pista,id_tiempo,Spotify Streams,YouTube Views,Track Score,Pandora Streams,TikTok Views,Shazam Counts,Track,Artist,Explicit Track,Release Date,dia,mes,año
61,59,55,1384.0,222076.0,186.5,NaN,8.811417e+09,NaN,Very Cute Melody by Marimba Tone (39813),mitsu sound,0,2023-12-31,31,12,2023
94,92,84,649404552.0,433839212.0,155.9,22603598.0,1.060356e+07,1380069.0,La Diabla,Xavi,0,2023-12-02,2,12,2023
97,95,87,359803580.0,369321334.0,153.1,20058510.0,8.740562e+08,1002835.0,BELLAKEO,Peso Pluma,1,2023-12-07,7,12,2023
131,129,112,8327.0,NaN,130.7,NaN,7.331825e+07,522.0,Looking Like a Wow End of Year 2023,WasteLand,0,2023-12-10,10,12,2023
177,174,152,606124553.0,166363445.0,112.6,1550124.0,2.960124e+07,1704574.0,LUNA,Feid,0,2023-12-01,1,12,2023


RETO 9


mes
1     3.652168e+11
5     1.779293e+11
11    1.734614e+11
10    1.672667e+11
3     1.638041e+11
7     1.503832e+11
6     1.484293e+11
4     1.462846e+11
8     1.426826e+11
9     1.363626e+11
2     1.333565e+11
12    1.022498e+11
Name: Spotify Streams, dtype: float64

RETO 10


,id_pista,id_tiempo,Spotify Streams,YouTube Views,Track Score,Pandora Streams,TikTok Views,Shazam Counts,Track,Artist,Explicit Track,Release Date,dia,mes,año
16,17,17,395433400.0,104673836.0,297.6,9961769.0,207317000.0,1210029.0,Fortnight (feat. Post Malone),Taylor Swift,0,2024-04-18,18,4,2024
866,861,17,166168092.0,16423009.0,52.3,2890486.0,7202700.0,116673.0,Whoï¿½ï¿½ï¿½s Afraid of Little O,Taylor Swift,0,2024-04-18,18,4,2024
1013,1007,17,175613480.0,15237337.0,47.8,2107786.0,1510800.0,67886.0,"So Long, London",Taylor Swift,0,2024-04-18,18,4,2024
1167,1159,17,161916445.0,13509432.0,44.0,2674527.0,NaN,47529.0,My Boy Only Breaks His Favorite Toys,Taylor Swift,0,2024-04-18,18,4,2024
1270,1260,17,165611453.0,9873283.0,42.1,1431013.0,6383500.0,79964.0,Guilty as Sin?,Taylor Swift,0,2024-04-18,18,4,2024
2484,2454,183,97601615.0,5528058.0,28.6,1393596.0,17893741.0,NaN,So High School,Taylor Swift,0,2024-04-19,19,4,2024
2517,2486,17,106201248.0,5521347.0,28.3,2070315.0,NaN,24502.0,The Alchemy,Taylor Swift,0,2024-04-18,18,4,2024
2528,2497,17,250430303.0,NaN,28.2,NaN,NaN,NaN,I Can Do It With a Broken Heart,Taylor Swift,0,2024-04-18,18,4,2024
2776,2744,183,88244010.0,7381186.0,26.7,1597647.0,14644000.0,32559.0,imgonnagetyouback,Taylor Swift,0,2024-04-19,19,4,2024
2816,2784,17,105424914.0,7227379.0,26.5,1332898.0,2500000.0,21372.0,Fresh Out The Slammer,Taylor Swift,0,2024-04-18,18,4,2024
